# 5-3절 연습 문제 풀이

이 노트북은 5-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch05/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
DATA_ROOT = '../../downloads'

def loaders(dataset='CIFAR10', batch_size=64, transform=None, train_transform=None):
    cls = getattr(datasets, dataset)
    transform = transform or transforms.ToTensor()
    full = cls(root=DATA_ROOT, train=True, download=True,
               transform=train_transform or transform)
    test_set = cls(root=DATA_ROOT, train=False, download=True, transform=transform)
    n_valid = int(len(full) * 0.2)
    g = torch.Generator().manual_seed(SEED)
    tr, va = random_split(full, [len(full) - n_valid, n_valid], generator=g)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=batch_size),
            DataLoader(test_set, batch_size=batch_size))

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot = correct = n = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            tot += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return tot / n, correct / n * 100

def fit(model, epochs=10, lr=1e-3, dataset='CIFAR10', **kw):
    tr, va, te = loaders(dataset, **kw)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        trl, _ = run_epoch(model, tr, criterion, optimizer)
        val, vaa = run_epoch(model, va, criterion)
        print(f'  {e}/{epochs} 훈련 {trl:.4f} / 검증 {val:.4f} ({vaa:.2f}%)')
    print(f'  평가 정확도 {run_epoch(model, te, criterion)[1]:.2f}%')

CIFAR_CLASSES = ['비행기', '자동차', '새', '고양이', '사슴',
                 '개', '개구리', '말', '배', '트럭']

def make_cifar_model(n_blocks=2, dropout=0.5):
    layers, fan_in, size = [], 3, 32
    for i in range(n_blocks):
        fan_out = 32 * (2 ** i)
        layers += [nn.Conv2d(fan_in, fan_out, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2)]
        fan_in, size = fan_out, size // 2
    layers += [nn.Flatten(), nn.Linear(fan_in * size * size, 128), nn.ReLU(),
               nn.Dropout(dropout), nn.Linear(128, 10)]
    return nn.Sequential(*layers)

## 연습 5-9

CIFAR-10 분류기 모델을 사용해 깃허브 저장소의 data 디렉터리에 있는 cat.jpg 파일 속 이미지의 클래스를 예측해 보자. 이미지 파일은 Pillow 라이브러리의 PIL.Image.open('cat.jpg') 함수를 호출해 Pillow 객체로 불러올 수 있다. 불러온 Pillow 객체는 torchvision.transforms의 변환 기능을 사용해 CIFAR-10 분류기 모델에 입력할 수 있는 형태로 변환해 사용해야 한다. 모델의 예측 결과가 예상과 다르다면 그 이유가 무엇인지 추측해 보자.

In [ ]:
from PIL import Image

# 1) CIFAR-10 분류기를 학습한다(시간 절약을 위해 에포크를 줄였다).
torch.manual_seed(SEED)
model = make_cifar_model()
fit(model, epochs=10)

In [ ]:
# 2) cat.jpg를 모델 입력 형태로 변환해 예측한다.
img = Image.open('../../data/cat.jpg').convert('RGB')
transform = transforms.Compose([
    transforms.Resize((32, 32)),      # CIFAR-10과 같은 크기로
    transforms.ToTensor(),
])
x = transform(img).unsqueeze(0).to(device)      # (1, 3, 32, 32)

model.eval()
with torch.no_grad():
    probs = torch.softmax(model(x), dim=1).squeeze()
top3 = probs.topk(3)
print(f'입력 이미지 형태: {tuple(x.shape)}')
for p, i in zip(top3.values.tolist(), top3.indices.tolist()):
    print(f'  {CIFAR_CLASSES[i]:6s} {p * 100:5.2f}%')
viz.plot_images([x.squeeze().cpu()], [f'예측: {CIFAR_CLASSES[top3.indices[0]]}'],
                images_per_row=1)

외부 이미지를 쓸 때는 **학습 데이터와 똑같은 전처리**를 거쳐야 한다. 크기를 32×32로 맞추고, `ToTensor()`로 0~1 범위의 (C, H, W) 텐서로 바꾼 뒤, 배치 차원을 추가한다.

학습 데이터에 정규화를 적용했다면 같은 정규화를 여기에도 적용해야 한다.

## 연습 5-10

[코드 5-17]의 완전 연결 계층에는 드롭아웃이 적용되어 있다.

만약 드롭아웃의 p 인자의 값을 0.5에서 0.9로 극단적으로 높인다면, 학습 과정과 모델의 최종 성능에 어떤 영향을 미칠지 추측해 보자.

이 드롭아웃은 분류기의 은닉층과 출력층 사이에 적용되어 있는데, 이는 일반적으로 드롭아웃이 적용되는 위치이다. 드롭아웃을 합성곱 계층이 아닌 완전 연결 계층의 은닉층과 출력층 사이에 주로 적용하는 이유를 '특징의 복잡도' 관점에서 설명해 보자.

In [ ]:
for p in (0.5, 0.9):
    torch.manual_seed(SEED)
    print(f'드롭아웃 p={p}')
    fit(make_cifar_model(dropout=p), epochs=10)

**p를 0.9로 높이면**: 학습할 때마다 은닉 뉴런의 90%가 꺼지므로 남는 정보가 너무 적다. 훈련 손실이 잘 떨어지지 않고 학습이 느려지며, 과적합은 줄지만 **과소적합**으로 기울어 최종 성능이 오히려 떨어진다.

**드롭아웃을 완전 연결 계층에 주로 적용하는 이유**
합성곱 계층은 필터 하나를 이미지 전체에 **공유**해 사용하므로 파라미터가 적고, 이미 구조적으로 과적합에 강하다. 반면 완전 연결 계층은 모든 뉴런이 모든 입력과 연결되어 파라미터가 압도적으로 많고 그만큼 과적합에 취약하다. 파라미터가 몰려 있는 곳에 규제를 거는 것이 효율적이다.

또한 합성곱 특징 지도에서 픽셀을 무작위로 끄면 인접 픽셀의 정보가 남아 있어 규제 효과가 약하다(그래서 합성곱에는 `Dropout2d`처럼 채널 단위로 끄는 변형을 쓴다).

## 연습 5-11

[코드 5-17]의 모델은 합성곱 계층, ReLU 활성화 계층, 최대 풀링 계층으로 구성된 합성곱 블록을 두 개 사용해 만들었다. 그런데 합성곱 블록의 수와 모델 성능의 관계는 어떻게 될까? kernel_size=3, stride=1, padding=1인 합성곱 블록 하나를 가진 CIFAR-10 분류 모델을 만들어 본 후, 합성곱 블록을 다섯 개까지 늘려보면서 모델 성능을 확인해 보자.

In [ ]:
for n_blocks in (1, 2, 3, 4, 5):
    torch.manual_seed(SEED)
    model = make_cifar_model(n_blocks)
    n_param = sum(p.numel() for p in model.parameters())
    print(f'합성곱 블록 {n_blocks}개 (파라미터 {n_param:,}개)')
    fit(model, epochs=10)
    print()

블록이 늘수록 특징 지도가 32 → 16 → 8 → 4 → 2 → 1로 줄고 채널은 늘어난다. 보통 **2~3개**까지는 성능이 오르지만 그 이상에서는 정체되거나 떨어진다.

특징 지도가 1×1까지 작아지면 공간 정보가 사라지고, 층이 깊어질수록 기울기 전달도 어려워지기 때문이다. 이 한계를 극복하는 방법이 8장의 배치 정규화와 잔차 연결이다.

## 연습 5-12

torchvision.transforms에 포함된 주요 이미지 변환 기능을 소개하는 4-3절의 [표 4-3]에는 이번 절에서 사용한 RandomHorizontalFlip(좌우 반전) 이외에도 RandomCrop(이미지 일부를 무작위 추출), ColorJitter(이미지 밝기, 대비, 채도, 색조를 무작위로 변경) 증강 변환도 포함되어 있다.

다른 두 증강 변환을 CIFAR-10 객체 분류기 학습에 사용하면 모델 성능이 어떻게 바뀔지 예상해 보자.

MNIST 숫자 분류기 모델에 좌우 반전을 포함한 증강 변환을 사용하면 모델 성능이 어떻게 바뀔지 예상해 보자.

In [ ]:
augmentations = {
    '증강 없음': transforms.ToTensor(),
    '좌우 반전': transforms.Compose([
        transforms.RandomHorizontalFlip(), transforms.ToTensor()]),
    '무작위 자르기': transforms.Compose([
        transforms.RandomCrop(32, padding=4), transforms.ToTensor()]),
    '색상 변형': transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor()]),
}
for name, tf in augmentations.items():
    torch.manual_seed(SEED)
    print(name)
    fit(make_cifar_model(), epochs=10, train_transform=tf)
    print()

증강 변환은 **훈련 데이터셋에만** 적용한다. 검증·평가는 항상 같은 조건이어야 비교가 가능하기 때문이다.

`RandomCrop(32, padding=4)`은 테두리를 4픽셀 채운 뒤 32×32로 잘라내 물체의 위치를 흔든다. `ColorJitter`는 조명 조건 변화를 흉내 낸다. 두 변환 모두 CIFAR-10처럼 사진 데이터에서 효과가 좋은데, 실제 사진이 이런 변형을 자연스럽게 포함하기 때문이다.

## 연습 5-13

[도전 문제] 이번 장에서 CIFAR-10 데이터셋 분류 모델의 성능을 73% 가까이 올려 보았다. 여러 방법을 동원해 분류 성능을 어디까지 올릴 수 있는지 도전해 보자. 다음과 같은 방법을 적용해 봐도 좋지만, 이외의 방법도 환영한다.

또 다른 데이터 증강 기법 적용

합성곱 계층의 수 또는 필터 수 조절

특징 지도의 크기 조절

분류기를 구성하는 다층 퍼셉트론 구조 변경

In [ ]:
# 여러 기법을 함께 적용한 강화 모델
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

def conv_bn(fan_in, fan_out):
    return nn.Sequential(nn.Conv2d(fan_in, fan_out, 3, 1, 1),
                         nn.BatchNorm2d(fan_out), nn.ReLU())

torch.manual_seed(SEED)
strong = nn.Sequential(
    conv_bn(3, 64), conv_bn(64, 64), nn.MaxPool2d(2),      # 32 -> 16
    conv_bn(64, 128), conv_bn(128, 128), nn.MaxPool2d(2),  # 16 -> 8
    conv_bn(128, 256), conv_bn(256, 256), nn.MaxPool2d(2), # 8 -> 4
    nn.Flatten(), nn.Dropout(0.5), nn.Linear(256 * 4 * 4, 10),
)
print(f'파라미터 {sum(p.numel() for p in strong.parameters()):,}개')
fit(strong, epochs=30, train_transform=train_tf, transform=test_tf)

성능을 끌어올리는 데 효과가 큰 순서는 대체로 다음과 같다.

1. **데이터 증강**(자르기 + 반전) — 가장 비용 대비 효과가 크다
2. **배치 정규화** — 학습이 안정되어 더 깊은 모델을 쓸 수 있다(8장)
3. **채널 수 확대와 블록당 합성곱 2개** — 표현력 강화
4. **정규화(Normalize)** — 입력 분포를 고르게 맞춘다

이 구성으로 30 에포크 학습하면 대체로 85% 안팎까지 오른다. 더 높이려면 학습률 스케줄러, 잔차 연결(8장), 사전 학습 모델의 전이 학습(8-4절)을 활용한다.

## 연습 5-14

[도전 문제] 어쩌다 보니 인류의 절반이 좌우 반전된 형태의 숫자를 사용하게 되었다. MNIST 데이터셋으로 모든 인류가 사용하는 숫자를 동시에 분류할 수 있는 숫자 분류기 모델을 만들어 보자.

5장 학습 노트

In [ ]:
# 좌우 반전된 숫자까지 함께 분류하는 모델
# 핵심: 훈련 데이터에 좌우 반전 이미지를 섞어 두 형태를 모두 학습시킨다.
flip_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),    # 절반은 반전된 형태로 학습
    transforms.ToTensor(),
])
torch.manual_seed(SEED)
model = nn.Sequential(
    nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10),
)
fit(model, epochs=10, dataset='MNIST', train_transform=flip_tf,
    transform=transforms.ToTensor())

In [ ]:
# 정상 이미지와 반전 이미지 각각에 대한 정확도를 따로 확인한다.
test_normal = datasets.MNIST(root=DATA_ROOT, train=False, download=True,
                             transform=transforms.ToTensor())
test_flip = datasets.MNIST(root=DATA_ROOT, train=False, download=True,
    transform=transforms.Compose([transforms.RandomHorizontalFlip(p=1.0),
                                  transforms.ToTensor()]))
criterion = nn.CrossEntropyLoss()
for name, ds in [('정상 숫자', test_normal), ('좌우 반전 숫자', test_flip)]:
    loader = DataLoader(ds, batch_size=128)
    print(f'{name}: {run_epoch(model, loader, criterion)[1]:.2f}%')

좌우 반전을 훈련 데이터에 섞으면 모델이 두 형태를 같은 클래스로 학습한다. 다만 주의할 점이 있다.

- **2와 5**처럼 좌우 반전하면 서로 닮아지는 숫자가 있어 정확도가 다소 떨어진다.
- 그래서 반전 여부까지 함께 맞히는 것이 목적이라면, 클래스를 20개(정상 10 + 반전 10)로 두는 설계가 더 낫다.

이 문제는 '데이터 증강으로 불변성을 학습시킨다'는 아이디어와, **불변성이 오히려 정보를 지우는 경우**가 있다는 점을 함께 보여 준다.